# DPO v4 on Google Colab

Trains DPO on top of the **SFT model** (called "v1" in the code). Plan and reasons: `finetune/DPO_V3.md` §7.
The pairs are built and audited in the repo, so this notebook only trains and exports.

### Files to upload to Colab (left file pane):
1. `adapters.safetensors`: the SFT LoRA adapter (from `runs/finetune/adapters/adapters.safetensors`, or the download
   link in `finetune/dpo/README.md`)
2. `train.jsonl` and `valid.jsonl` from `finetune/dpo/v4/`, rebuilt with `uv run python -m finetune.build_dpo_pairs_v4 --seed 42`

Pick a GPU runtime (Runtime > Change runtime type), then Run all.

### Outputs
- `qwen2.5-3b-haggle-dpo-v4-q4_k_m.gguf` and its SHA-256
- `training_log.json` (versions, settings, every logged step including the reward columns and eval)

## 1. Install Dependencies
Installs pre-built wheels only (no compiling, takes ~15s).

In [ ]:
# Colab preinstalls broken torchao 0.10.0; remove it so PEFT uses standard PyTorch LoRA
!pip uninstall -y torchao
# Use Colab's own torch/transformers and the latest TRL. Do NOT pin an old trl: old trl needs old transformers,
# and old DPOConfig has gradient checkpointing off by default, which runs a 16 GB GPU out of memory.
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl safetensors gguf protobuf

## 2. Check Uploads & Convert MLX Adapter to PEFT

In [ ]:
import json, os, torch
from safetensors.torch import load_file, save_file

for f in ("adapters.safetensors", "train.jsonl", "valid.jsonl"):
    assert os.path.exists(f), f"Upload {f} to the Colab file browser!"

# Convert MLX weight matrices [in_dim, r] to PyTorch PEFT layout [r, in_dim]
print("Converting MLX adapter to PyTorch PEFT layout...")
mlx_weights = load_file("adapters.safetensors")
peft_dict = {}
for k, v in mlx_weights.items():
    if k == "__metadata__":
        continue
    new_k = "base_model.model." + k
    if new_k.endswith(".lora_a"):
        new_k = new_k[:-7] + ".lora_A.weight"
        peft_dict[new_k] = v.T.contiguous()
    elif new_k.endswith(".lora_b"):
        new_k = new_k[:-7] + ".lora_B.weight"
        peft_dict[new_k] = v.T.contiguous()
    else:
        peft_dict[new_k] = v

os.makedirs("v1_peft_adapter", exist_ok=True)
save_file(peft_dict, "v1_peft_adapter/adapter_model.safetensors")

peft_config = {
    "base_model_name_or_path": "Qwen/Qwen2.5-3B-Instruct",
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layers_pattern": None,
    "layers_to_transform": None,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 16,
    "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "task_type": "CAUSAL_LM"
}
with open("v1_peft_adapter/adapter_config.json", "w") as f:
    json.dump(peft_config, f, indent=2)

print("Adapter successfully prepared in ./v1_peft_adapter")

## 3. Load Base Model and Fuse v1 SFT Weights

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Downloading base Qwen-3B...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=REVISION,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Attaching and merging v1 adapter...")
v1_model = PeftModel.from_pretrained(base_model, "./v1_peft_adapter")
v1_fused = v1_model.merge_and_unload()
print("Full v1 model assembled and ready in GPU VRAM!")

## 4. Load DPO Preference Dataset

In [ ]:
from datasets import Dataset

def load_pairs(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                rows.append({
                    "prompt": tokenizer.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=True),
                    "chosen": item["chosen"],
                    "rejected": item["rejected"],
                })
    return Dataset.from_list(rows)

dpo_dataset = load_pairs("train.jsonl")
eval_dataset = load_pairs("valid.jsonl")
print(f"Loaded {len(dpo_dataset)} training pairs and {len(eval_dataset)} held-out pairs.")
assert len(dpo_dataset) > 3500, "Expected the v4 file (about 4,100 pairs). Did you upload finetune/dpo/v4/train.jsonl?"

# Every pair must fit whole. The floor is near the start of the prompt and the decision is the reply at the end,
# so truncating either end removes the one thing DPO has to learn (v4: 14% of pairs exceed 1,024 tokens).
def n_tokens(row):
    p = len(tokenizer(row["prompt"], add_special_tokens=False)["input_ids"])
    return p + max(len(tokenizer(row[k], add_special_tokens=False)["input_ids"]) for k in ("chosen", "rejected")) + 1
longest = max(n_tokens(r) for d in (dpo_dataset, eval_dataset) for r in d)
MAX_LENGTH = 1536
print(f"Longest pair: {longest} tokens (limit {MAX_LENGTH})")
assert longest <= MAX_LENGTH, "A pair is longer than MAX_LENGTH and would be cut: raise MAX_LENGTH."

## 5. Run DPO Training

In [ ]:
import inspect
import peft, transformers, trl
from trl import DPOTrainer, DPOConfig

VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "trl": trl.__version__}
print(VERSIONS)

dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

dpo_config_params = inspect.signature(DPOConfig.__init__).parameters
# v4 needs the SFT term on the chosen reply (keeps good replies likely while DPO pushes bad ones down).
# Old TRL calls it rpo_alpha; TRL 1.x removed rpo_alpha and adds it as a second loss, loss_type=["sigmoid", "sft"].
# v3 set rpo_alpha only "if supported", so on TRL 1.x it was silently skipped: here one of the two must apply.
if "rpo_alpha" in dpo_config_params:
    SFT_TERM = {"rpo_alpha": 1.0}
else:
    SFT_TERM = {"loss_type": ["sigmoid", "sft"], "loss_weights": [1.0, 1.0]}
config_kwargs = {
    "output_dir": "./dpo_checkpoints",
    "learning_rate": 5e-6,
    "beta": 0.1,
    **SFT_TERM,
    # Same effective batch as v3 (8), but 1 pair per step: pairs are now up to 1,536 tokens, and with Qwen's
    # 152k vocabulary the logits alone are several GB at batch 2.
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "per_device_eval_batch_size": 1,
    "gradient_checkpointing": True,  # TRL 1.x default; old TRL defaults to False, which is what ran out of memory
    "num_train_epochs": 1,
    "fp16": True,
    "seed": 42,
    "logging_steps": 25,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 6,
    "eval_steps": 100,
    "remove_unused_columns": False,
    "report_to": "none",
}
# The eval switch was renamed across transformers versions.
config_kwargs["eval_strategy" if "eval_strategy" in dpo_config_params else "evaluation_strategy"] = "steps"
if "max_length" in dpo_config_params:
    config_kwargs["max_length"] = MAX_LENGTH
if "max_prompt_length" in dpo_config_params:  # old TRL only; it cut prompts from the START, deleting the floor
    config_kwargs["max_prompt_length"] = MAX_LENGTH

training_args = DPOConfig(**config_kwargs)
if "rpo_alpha" in SFT_TERM:
    assert training_args.rpo_alpha == 1.0
else:
    assert "sft" in training_args.loss_type, training_args.loss_type
print("SFT term:", SFT_TERM)
v1_fused.config.use_cache = False  # needed with gradient checkpointing

trainer_params = inspect.signature(DPOTrainer.__init__).parameters
trainer_kwargs = {
    "model": v1_fused,
    "ref_model": None,
    "args": training_args,
    "peft_config": dpo_lora_config,
    "train_dataset": dpo_dataset,
    "eval_dataset": eval_dataset,
}
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "max_length" in trainer_params:
    trainer_kwargs["max_length"] = MAX_LENGTH
if "max_prompt_length" in trainer_params:
    trainer_kwargs["max_prompt_length"] = MAX_LENGTH

trainer = DPOTrainer(**trainer_kwargs)

print("Starting DPO training...")
trainer.train()
print("DPO training complete!")

## 5b. Save the training log

In [ ]:
# Save what the report needs: versions, settings and every logged step.
# Warning signs (finetune/DPO_V3.md section 3): rewards/chosen falling steadily below 0, or rewards/accuracies near 1.0 early.
log = {"versions": VERSIONS, "config": dict(config_kwargs),
       "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "targets": ["q_proj", "k_proj", "v_proj", "o_proj"]},
       "sft_term": SFT_TERM, "longest_pair_tokens": longest, "n_train": len(dpo_dataset), "n_valid": len(eval_dataset), "log_history": trainer.state.log_history}
with open("training_log.json", "w") as f:
    json.dump(log, f, indent=1, default=str)
for row in trainer.state.log_history:
    keys = [k for k in ("loss", "rewards/chosen", "rewards/rejected", "rewards/accuracies", "rewards/margins", "nll_loss", "sft_loss",
                        "eval_loss", "eval_rewards/accuracies") if k in row]
    print(row.get("step"), {k: round(row[k], 3) for k in keys})

## 6. Merge & Quantize to GGUF (for Ollama)

In [ ]:
print("Fusing DPO weights...")
final_model = trainer.model.merge_and_unload()
OUT = "qwen2.5-3b-haggle-dpo-v4"
final_model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print(f"Saved fused HF model to ./{OUT}")

In [ ]:
%%bash
set -e
OUT="qwen2.5-3b-haggle-dpo-v4"
[ -d llama.cpp ] || git clone -q --branch b6000 --depth 1 https://github.com/ggml-org/llama.cpp
python llama.cpp/convert_hf_to_gguf.py "$OUT" --outtype f16 --outfile "$OUT-f16.gguf"
cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
cmake --build llama.cpp/build --target llama-quantize -j > /dev/null
llama.cpp/build/bin/llama-quantize "$OUT-f16.gguf" "$OUT-q4_k_m.gguf" q4_k_m
rm "$OUT-f16.gguf"
echo "Export complete!"
ls -lh "$OUT-q4_k_m.gguf"
sha256sum "$OUT-q4_k_m.gguf"

## 7. Download
Download `qwen2.5-3b-haggle-dpo-v4-q4_k_m.gguf` (about 1.9 GB) and `training_log.json`, and keep the SHA-256 printed
above. The checkpoints in `./dpo_checkpoints` (every 100 steps) are kept in case the final model overshoots: the
decision check (`python -m finetune.check_decisions`) decides whether an earlier one is needed.